In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

### Set time resolution

In [3]:
import re 

# define time resolution
timeres = '30min'
## calculate period based on 1 day
number = int(re.sub(r'\D', '', timeres)) # \D matches any non-digit character
# check if time resolution is in minutes
min_flag = 'min' in timeres
if min_flag:
    mins = int(number)
    # calculate hour fraction
    period = int((60 / mins) * 24)
    path = f'../data/{timeres}_sfl.csv'
    # calculate period (how many hours in a day)
else: # otherwise, periodicty is hourly
    period = int((24 / number))
    path = '../data/hourly_sfl.csv'

# calculate 1 hour offset from period
hour_offset = int(period / 24)
pops = ['prochloro', 'synecho', 'picoeuk']

### Load data

In [ ]:
# read in smoothed PAR data
hot_cruise_smooth_all = pd.read_pickle(f'../data/smoothed_par_data/hot_cruise_smooth_{timeres}.pickle')
hot_cruise_smooth = hot_cruise_smooth_all.loc[hot_cruise_smooth_all['qc_flag'] == 0]

## Full cruise testing
Instead of calculating growth from each day, calculate throughout entire cruise then split up by day

In [ ]:
from scipy.signal import savgol_filter
from statsmodels.nonparametric.smoothers_lowess import lowess

cruise = "KM1513"
pop = "prochloro"

window_length = 9
polyorder = 3
lowess_neighbors = 9

# Half-hourly observations: split whenever a sample is missing.
gap_threshold = pd.Timedelta(minutes=45)

data = hot_cruise_smooth.loc[
    (hot_cruise_smooth["cruise"] == cruise)
    & (hot_cruise_smooth["pop"] == pop)
].copy()

# Retains the supplied clock times; no manual hour adjustment.
data["time"] = pd.to_datetime(data["time"], errors="coerce")
data["log_diel"] = pd.to_numeric(
    data["log_diel"], errors="coerce"
)
data["log_diel"] = data["log_diel"].replace(
    [np.inf, -np.inf], np.nan
)

data = (
    data.dropna(subset=["time", "log_diel"])
    .sort_values("time")
    .reset_index(drop=True)
)

if data.empty:
    raise ValueError("No valid observations for this cruise/population.")

if data.duplicated(subset = ['cruise_day', 'time']).any():
    raise ValueError(
        "Duplicate timestamps found. Resolve these before smoothing."
    )

# Identify uninterrupted, approximately half-hourly segments.
data["segment"] = (
    data["time"].diff().gt(gap_threshold).cumsum()
)

data["savgol_diel"] = np.nan
data["lowess_diel"] = np.nan
data["sg_applied"] = False

for _, segment in data.groupby("segment", sort=False):
    idx = segment.index
    y = segment["log_diel"].to_numpy(dtype=float)

    elapsed_hours = (
        (segment["time"] - segment["time"].iloc[0])
        .dt.total_seconds()
        .to_numpy(dtype=float)
        / 3600
    )

    # Savitzky–Golay: shorten the window for short segments.
    w = min(window_length, len(segment))
    if w % 2 == 0:
        w -= 1

    if w > polyorder:
        data.loc[idx, "savgol_diel"] = savgol_filter(
            y,
            window_length=w,
            polyorder=polyorder,
            mode="interp",
        )
        data.loc[idx, "sg_applied"] = True
    else:
        # Too few points to apply the requested polynomial smoother.
        data.loc[idx, "savgol_diel"] = y

    # LOWESS: keep the neighborhood approximately fixed in point count.
    if len(segment) >= 4:
        frac = min(1.0, lowess_neighbors / len(segment))

        data.loc[idx, "lowess_diel"] = lowess(
            endog=y,
            exog=elapsed_hours,
            frac=frac,
            it=0,
            return_sorted=False,
        )
    else:
        data.loc[idx, "lowess_diel"] = y


# --------------------------------------------------------------
# Plot without drawing lines across sampling gaps.
# --------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 5))

ax.scatter(
    data["time"],
    data["log_diel"],
    color="tab:blue",
    s=16,
    alpha=0.55,
    label="Observed",
)

for i, (_, segment) in enumerate(
    data.groupby("segment", sort=False)
):
    ax.plot(
        segment["time"],
        segment["savgol_diel"],
        color="red",
        linewidth=1.8,
        label=f"Savitzky–Golay ({window_length}, {polyorder})"
        if i == 0 else None,
    )

    ax.plot(
        segment["time"],
        segment["lowess_diel"],
        color="g",
        linestyle="--",
        linewidth=1.5,
        label=f"LOWESS (~{lowess_neighbors} neighbors)"
        if i == 0 else None,
    )

locator = mdates.AutoDateLocator()
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(
    mdates.ConciseDateFormatter(locator)
)

ax.set_title(f"{cruise} | {pop} | Diel smoothing")
ax.set_xlabel("Sampling date/time")
ax.set_ylabel("log_diel")
ax.margins(x=0.01)
ax.legend()

fig.tight_layout()
plt.show()


# --------------------------------------------------------------
# Compare daily extrema and ranges.
# Uses your existing cruise-day definition, not UTC calendar days.
# --------------------------------------------------------------
daily_comparison = data.groupby("cruise_day").agg(
    n=("log_diel", "size"),
    observed_min=("log_diel", "min"),
    observed_max=("log_diel", "max"),
    sg_min=("savgol_diel", "min"),
    sg_max=("savgol_diel", "max"),
    lowess_min=("lowess_diel", "min"),
    lowess_max=("lowess_diel", "max"),
)

daily_comparison["observed_range"] = (
    daily_comparison["observed_max"]
    - daily_comparison["observed_min"]
)

for method in ["sg", "lowess"]:
    daily_comparison[f"{method}_range"] = (
        daily_comparison[f"{method}_max"]
        - daily_comparison[f"{method}_min"]
    )

    daily_comparison[f"{method}_range_ratio"] = (
        daily_comparison[f"{method}_range"]
        / daily_comparison["observed_range"].replace(0, np.nan)
    )

print(daily_comparison.round(4))

In [ ]:
data.to_csv('~/diel_cruise_filter_compare.csv')

In [ ]:
pop = 'prochloro'
c = 'tab:blue'
cruise = 'KM1513'
diel_cruise = hot_cruise_smooth.loc[hot_cruise_smooth['cruise'] == cruise]
# test with pro only
pop_df = diel_cruise.loc[
            diel_cruise["pop"] == pop
        ].copy()

# Remove missing values and ensure chronological ordering.
data = (
    pop_df.dropna(subset=["log_diel"])
    .sort_values(["time"])
    .copy()
).reset_index(drop = True)
# skip if empty
if data.empty:
    print("Empty")
    #continue

reset_x = data.index.to_numpy(dtype=float)
y = data["log_diel"].to_numpy(dtype=float)

# Ensure an odd window that does not exceed the number of points.
this_window = min(window_length, len(y))

if this_window % 2 == 0:
    this_window -= 1

# Apply Savitzky–Golay only when enough points are available.
if this_window > polyorder:
    sg_smoothed = savgol_filter(
        y,
        window_length=this_window,
        polyorder=polyorder,
        mode="interp",
    )
else:
    sg_smoothed = y.copy()

data["savgol_diel"] = sg_smoothed

# also fit lowess curve  
frac = 0.43
frac_cruise = frac * (24 / len(data))
smoothed = sm.nonparametric.lowess(endog=y, exog=reset_x, frac=frac_cruise)
# add lowess results to plot
data['lowess_diel'] = smoothed[:, 1]

smoothed_dfs.append(data)

# Optional: retain your quadratic fit for comparison.
if len(y) >= 3:
    z = np.polyfit(reset_x, y, 2)
    quadratic_fit = np.poly1d(z)

# Plot the raw observations and Savitzky–Golay curve.
fig, ax = plt.subplots(figsize = (10, 6))
# plot data
ax.scatter(
    reset_x,
    y,
    color=c,
    label="Observed log_diel",
)
# plot savgol fit
ax.plot(
    reset_x,
    sg_smoothed,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Savitzky–Golay (window={this_window}, order={polyorder})",
)

# add LOWESS fit to plot
ax.plot(smoothed[:, 0], 
        smoothed[:, 1], 
        color='magenta', 
        linestyle='--', 
        linewidth=2, 
        label = 'Lowess Regression')


ax.set_title(f"Cruise {cruise} | Pop {pop} ")
# ax.set_xlim((-0.5, 14.5))
ax.set_xlabel("Hour")
ax.set_ylabel("log_diel")
ax.legend()
# plt.show()

In [ ]:
cruise = 'TN397'
pop = 'prochloro'

hot_cruise_smooth.loc[
    (hot_cruise_smooth["cruise"] == cruise)
    & (hot_cruise_smooth["pop"] == pop)